# Fall alerts in a care home: training pipeline

A camera image of a resident's room is checked for someone who has fallen. When the system
raises an alert, the night nurse goes to the room. **Falling is the positive class** throughout.

This notebook downloads the dataset, describes it, finetunes `yolo11n`, and evaluates it per
class on the test split. Baselines, the cost sweep and the human-review band follow below.

Run it end to end with `uv run python main.py train`, or on Colab with a T4 GPU.

## 0. Setup

On Colab: Runtime > Change runtime type > T4, and add a secret named `ROBOFLOW_API_KEY` (key
icon in the sidebar) with notebook access switched on. Locally the key comes from `.env`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab starts in an empty folder, so fetch the repo to get api_key.py and the weights/ folder.
    if not Path("api_key.py").exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Allan-Data-science/daiist-assignment-2.git"],
            check=True,
        )
        os.chdir("daiist-assignment-2")
    # Plain pip, not %pip: main.py runs this notebook as an ordinary script, where magics do not
    # exist. No uv sync either, because Colab brings its own GPU build of PyTorch. Ultralytics is
    # pinned to the same range as pyproject.toml so the weights load in the local app.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.4,<8.5", "roboflow"],
        check=True,
    )
    # api_key.py checks the environment variable first, so the Colab secret goes there.
    from google.colab import userdata

    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

from api_key import load_api_key

API_KEY = load_api_key()
# Print the shape of the key, never the key.
print(f"key loaded: {len(API_KEY)} characters")

## 1. The dataset

| | |
|---|---|
| workspace | `falldetection-bwfdz` |
| project | `fall-detection-ehgga` |
| version | `1` |
| classes | `Falling`, `Sitting`, `Standing` |
| licence | CC BY 4.0 |

The version number is pinned so the same images come back every time.

In [ ]:
import yaml
from roboflow import Roboflow

WORKSPACE, PROJECT, VERSION = "falldetection-bwfdz", "fall-detection-ehgga", 1
DATASET_DIR = Path(f"datasets/{PROJECT}-{VERSION}").resolve()

if not DATASET_DIR.exists():
    rf = Roboflow(api_key=API_KEY)
    rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download(
        "yolov11", location=str(DATASET_DIR)
    )

# Roboflow writes split paths like "../train/images", which point outside the dataset folder.
# Rewrite them relative to an explicit root so Ultralytics finds the images wherever this runs.
DATA_YAML = DATASET_DIR / "data.yaml"
data_config = yaml.safe_load(DATA_YAML.read_text())
data_config.update(
    path=str(DATASET_DIR), train="train/images", val="valid/images", test="test/images"
)
DATA_YAML.write_text(yaml.safe_dump(data_config))

CLASS_NAMES = data_config["names"]
POSITIVE = "Falling"
print(CLASS_NAMES)

### 1.1 Describe it before training on it

Instances per class, images per split, and how much of the image a person covers. Roboflow
augmented the training split, so it also counts how many *original* photographs are behind it:
augmented copies share the part of the file name before `.rf.`.

In [ ]:
from collections import Counter


def describe_split(split):
    label_files = sorted((DATASET_DIR / split / "labels").glob("*.txt"))
    instances, box_areas, empty = Counter(), [], 0
    for f in label_files:
        # A YOLO label file is one box per line: "class_id x y w h", all relative to the image.
        lines = [line.split() for line in f.read_text().splitlines() if line.strip()]
        if not lines:
            empty += 1
        for parts in lines:
            instances[CLASS_NAMES[int(parts[0])]] += 1
            box_areas.append(float(parts[3]) * float(parts[4]))
    originals = {f.name.split(".rf.")[0] for f in label_files}
    box_areas.sort()
    return {
        "images": len(label_files),
        "originals": len(originals),
        "empty": empty,
        **{name: instances[name] for name in CLASS_NAMES},
        "median_box": box_areas[len(box_areas) // 2],
    }


dataset_stats = {split: describe_split(split) for split in ("train", "valid", "test")}

print(f"{'split':<7}{'images':>8}{'originals':>11}{'empty':>7}", end="")
print("".join(f"{name:>10}" for name in CLASS_NAMES) + f"{'median box':>12}")
for split, s in dataset_stats.items():
    print(f"{split:<7}{s['images']:>8}{s['originals']:>11}{s['empty']:>7}", end="")
    print("".join(f"{s[name]:>10}" for name in CLASS_NAMES) + f"{s['median_box']:>12.0%}")

## 2. Finetune yolo11n

Start from the COCO-pretrained checkpoint, so the model already knows what a person looks like
and only has to learn the posture. `patience` stops training once validation mAP has not improved
for that many epochs, which guards against overfitting a training set of only 656 originals.

`EPOCHS` can be overridden from the environment for a quick smoke test (`EPOCHS=1`).

In [ ]:
import shutil

from ultralytics import YOLO

EPOCHS = int(os.environ.get("EPOCHS", 50))
IMGSZ, BATCH, PATIENCE, SEED = 640, 16, 10, 0

model = YOLO("yolo11n.pt")
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    project=str(Path("runs").resolve()),
    name="fall",
    exist_ok=True,
)

# runs/ is not committed. The app and the grader load weights/best.pt, so copy it there.
best_run_weights = Path(model.trainer.save_dir) / "weights" / "best.pt"
WEIGHTS = Path("weights/best.pt")
shutil.copy(best_run_weights, WEIGHTS)
print(f"saved {WEIGHTS} ({WEIGHTS.stat().st_size / 1e6:.1f} MB)")

## 3. Per-class results on the test split

The test split was never used for training or for choosing when to stop, so it is the honest
number. The overall mAP is printed, but the row that matters is `Falling`.

In [ ]:
import json

finetuned = YOLO(str(WEIGHTS))
test_metrics = finetuned.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, plots=True)
box = test_metrics.box

per_class = {
    name: {
        "precision": float(box.p[i]),
        "recall": float(box.r[i]),
        "ap50": float(box.ap50[i]),
        "ap50_95": float(box.maps[i]),
    }
    for i, name in enumerate(CLASS_NAMES)
}

print(f"overall  mAP50 {box.map50:.3f}   mAP50-95 {box.map:.3f}\n")
print(f"{'class':<10}{'precision':>11}{'recall':>9}{'AP50':>8}{'AP50-95':>10}")
for name, m in per_class.items():
    print(
        f"{name:<10}{m['precision']:>11.3f}{m['recall']:>9.3f}"
        f"{m['ap50']:>8.3f}{m['ap50_95']:>10.3f}"
    )

# Rows are predicted class, columns are true class; the last row/column is background.
confusion_matrix = test_metrics.confusion_matrix.matrix.astype(int).tolist()
print(f"\nconfusion matrix plot: {test_metrics.save_dir}/confusion_matrix.png")

# Everything the app needs later goes in outputs/, which is committed alongside the weights.
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)
(OUTPUTS / "test_metrics.json").write_text(
    json.dumps(
        {
            "dataset": dataset_stats,
            "overall": {"map50": float(box.map50), "map50_95": float(box.map)},
            "per_class": per_class,
            "confusion_matrix": confusion_matrix,
            "matrix_labels": CLASS_NAMES + ["background"],
            "train_settings": {"epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
                               "patience": PATIENCE, "seed": SEED},
        },
        indent=2,
    )
)

## 4. From boxes to one decision per image

The nurse does not act on boxes, she acts on one question per image: *has someone fallen?* Each
image shows one person, so **one image is one case**. Every system below is reduced to a single
**fall score** per image, and an alert is raised when the score is at or above a cut:

| system | fall score |
|---|---|
| finetuned yolo11n | confidence of the most confident `Falling` box, 0 if none |
| A. COCO, any person | confidence of the most confident `person` box, 0 if none |
| C. COCO, wide box (bonus) | most confident `person` box that is wider than tall, 0 if none |
| predict nothing | always 0, so never an alert |

Taking the maximum, not the mean: one confident fall box is enough evidence to send the nurse.

Predictions are kept down to 1% confidence, so the borderline cases are not thrown away before the
threshold sweep sees them.

In [ ]:
TEST_IMAGES = sorted((DATASET_DIR / "test" / "images").glob("*.jpg"))


def is_fall(image_path):
    # Ground truth for one image: does any label in it say Falling?
    label_path = DATASET_DIR / "test" / "labels" / f"{image_path.stem}.txt"
    ids = {int(line.split()[0]) for line in label_path.read_text().splitlines() if line.strip()}
    return CLASS_NAMES.index(POSITIVE) in ids


cases = [{"image": p.name, "truth": is_fall(p)} for p in TEST_IMAGES]
print(f"{len(cases)} test images, {sum(c['truth'] for c in cases)} with a fall")

In [ ]:
coco = YOLO("yolo11n.pt")  # the same checkpoint we finetuned from, used as it is
COCO_PERSON = 0  # "person" is class 0 in COCO
FALLING = CLASS_NAMES.index(POSITIVE)


def max_conf(boxes):
    return max((float(b.conf) for b in boxes), default=0.0)


for case, path in zip(cases, TEST_IMAGES):
    ours = finetuned.predict(path, conf=0.01, imgsz=IMGSZ, verbose=False)[0].boxes
    case["finetuned"] = max_conf([b for b in ours if int(b.cls) == FALLING])

    people = [b for b in coco.predict(path, conf=0.01, verbose=False)[0].boxes
              if int(b.cls) == COCO_PERSON]
    case["coco_person"] = max_conf(people)
    # xywh is centre x, centre y, width, height. Someone lying on the floor is wider than tall.
    case["coco_wide"] = max_conf([b for b in people if b.xywh[0][2] > b.xywh[0][3]])

    case["nothing"] = 0.0

SYSTEMS = ["finetuned", "coco_person", "coco_wide", "nothing"]

### 4.1 The four systems on the test split

Falling is positive. The table is at two example cuts only; the cut actually used is chosen by
cost in the next section, from the same scores.

In [ ]:
def confusion(cases, system, cut):
    # Positive class is a fall. A score at or above the cut raises an alert.
    tp = sum(1 for c in cases if c[system] >= cut and c["truth"])
    fp = sum(1 for c in cases if c[system] >= cut and not c["truth"])
    fn = sum(1 for c in cases if c[system] < cut and c["truth"])
    tn = sum(1 for c in cases if c[system] < cut and not c["truth"])
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "recall": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "precision": tp / (tp + fp) if tp + fp else float("nan"),
        "accuracy": (tp + tn) / len(cases),
    }


for cut in (0.25, 0.5):
    print(f"cut {cut}")
    print(f"{'system':<13}{'TP':>4}{'FP':>4}{'FN':>4}{'TN':>4}"
          f"{'recall':>9}{'specificity':>13}{'precision':>11}{'accuracy':>10}")
    for system in SYSTEMS:
        s = confusion(cases, system, cut)
        precision = "-" if s["precision"] != s["precision"] else f"{s['precision']:.1%}"
        print(f"{system:<13}{s['tp']:>4}{s['fp']:>4}{s['fn']:>4}{s['tn']:>4}"
              f"{s['recall']:>9.1%}{s['specificity']:>13.1%}{precision:>11}{s['accuracy']:>10.1%}")
    print()

# Per-image scores are what the cost sweep and the dashboard are built from.
(OUTPUTS / "test_scores.json").write_text(json.dumps(cases, indent=2))

## 5. Cost per 1,000 cases and the threshold sweep

The cost of each outcome, argued in REPORT.md:

| outcome | cost | what happens |
|---|---|---|
| **TP** | EUR 5 | nurse goes to the room and finds the fall |
| **FP** | EUR 5 | nurse goes to the room and finds no fall |
| **FN** | EUR 2,000 | fall missed, resident lies on the floor until the next round |
| **TN** | EUR 0 | no fall, no alert |

**Base rate: 1%** of the images the system checks contain a fall. The test split is 33% falls, so
its precision and accuracy do not transfer. Recall and specificity are properties of the model
and do, so they are measured on the test split and projected onto the real mix:

```
TP = p * recall                   FN = p * (1 - recall)
FP = (1 - p) * (1 - specificity)  TN = (1 - p) * specificity
```

In [ ]:
COST_TP, COST_FP, COST_FN, COST_TN = 5.0, 5.0, 2000.0, 0.0
BASE_RATE = 0.01

# Cuts from 0.01 to 1.01 (never alert) in steps of 0.01. A cut of 0 is left out: a score of 0
# means the model found no box at all, and alerting on that is "check every image", which is
# priced separately as the everyone-reviews design, not as a model threshold.
CUTS = [round(i / 100, 2) for i in range(1, 102)]


def cost_per_1000(stats, base_rate=BASE_RATE, cost_fn=COST_FN):
    falls, others = 1000 * base_rate, 1000 * (1 - base_rate)
    tp, fn = falls * stats["recall"], falls * (1 - stats["recall"])
    fp, tn = others * (1 - stats["specificity"]), others * stats["specificity"]
    return tp * COST_TP + fp * COST_FP + fn * cost_fn + tn * COST_TN


def sweep(system, base_rate=BASE_RATE, cost_fn=COST_FN):
    # One row per cut: the cut, its expected cost per 1,000 cases, and the test confusion.
    rows = []
    for cut in CUTS:
        s = confusion(cases, system, cut)
        rows.append({"cut": cut, "cost": cost_per_1000(s, base_rate, cost_fn), **s})
    return rows


def cheapest(rows):
    return min(rows, key=lambda r: r["cost"])


sweeps = {system: sweep(system) for system in SYSTEMS}

print(f"each system at its own cheapest cut, base rate {BASE_RATE:.0%}\n")
print(f"{'system':<13}{'cut':>6}{'recall':>9}{'specificity':>13}{'EUR/1,000':>12}")
for system in SYSTEMS:
    best = cheapest(sweeps[system])
    print(f"{system:<13}{best['cut']:>6}{best['recall']:>9.1%}"
          f"{best['specificity']:>13.1%}{best['cost']:>12,.2f}")

CHOSEN = cheapest(sweeps["finetuned"])
print(f"\nfinetuned model: cheapest single cut {CHOSEN['cut']} at EUR {CHOSEN['cost']:,.2f} per 1,000")

In [ ]:
import matplotlib

matplotlib.use("Agg")  # no window: main.py runs this as a plain script
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
for system in SYSTEMS:
    ax.plot([r["cut"] for r in sweeps[system]], [r["cost"] for r in sweeps[system]], label=system)
ax.scatter([CHOSEN["cut"]], [CHOSEN["cost"]], color="black", zorder=3,
           label=f"chosen cut {CHOSEN['cut']}")
ax.set_xlabel("confidence threshold (alert at or above)")
ax.set_ylabel("expected cost, EUR per 1,000 cases")
ax.set_title(f"Cost against threshold at a {BASE_RATE:.0%} base rate")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUTS / "cost_curve.png", dpi=120)

### 5.1 Sensitivity: how much does the answer depend on the two weakest assumptions?

The FN cost and the base rate are the least certain figures. Each is varied with everything else
held fixed, and the cheapest cut for the finetuned model is found again. The cheapest cut moving
is expected. What matters is the point where the decision itself changes: the system is no longer
worth running at all, or a baseline becomes cheaper.

In [ ]:
def best_by_system(base_rate=BASE_RATE, cost_fn=COST_FN):
    return {s: cheapest(sweep(s, base_rate, cost_fn)) for s in SYSTEMS}


def print_sensitivity(label, values, fmt, **fixed):
    print(f"{label:>12}{'best cut':>10}{'recall':>9}{'EUR/1,000':>12}"
          f"{'coco_wide':>12}{'nothing':>11}{'cheapest system':>17}")
    rows = []
    for value in values:
        best = best_by_system(**{**fixed, **value})
        ours = best["finetuned"]
        winner = min(best, key=lambda s: best[s]["cost"])
        print(f"{fmt(value):>12}{ours['cut']:>10}{ours['recall']:>9.1%}{ours['cost']:>12,.2f}"
              f"{best['coco_wide']['cost']:>12,.2f}{best['nothing']['cost']:>11,.2f}{winner:>17}")
        rows.append({**value, **{s: best[s]["cost"] for s in SYSTEMS},
                     "best_cut": ours["cut"], "winner": winner})
    print()
    return rows


FN_COSTS = [250, 500, 1000, 2000, 5000, 10000]
BASE_RATES = [0.001, 0.0025, 0.005, 0.01, 0.02, 0.05]

print(f"FN cost varied, base rate fixed at {BASE_RATE:.0%}")
fn_sensitivity = print_sensitivity(
    "FN cost", [{"cost_fn": c} for c in FN_COSTS], lambda v: f"EUR {v['cost_fn']:,}"
)
print(f"base rate varied, FN cost fixed at EUR {COST_FN:,.0f}")
rate_sensitivity = print_sensitivity(
    "base rate", [{"base_rate": r} for r in BASE_RATES], lambda v: f"{v['base_rate']:.2%}"
)

In [ ]:
(OUTPUTS / "cost_sweep.json").write_text(
    json.dumps(
        {
            "costs": {"tp": COST_TP, "fp": COST_FP, "fn": COST_FN, "tn": COST_TN},
            "base_rate": BASE_RATE,
            "sweeps": sweeps,
            "chosen": CHOSEN,
            "fn_sensitivity": fn_sensitivity,
            "rate_sensitivity": rate_sensitivity,
        },
        indent=2,
    )
)

## 6. Pass, human review, fail

To do: the two thresholds, review load and staffing.